# 🎨 See-through — Anime Layer Decomposition
**SIGGRAPH 2026** | 1枚のアニメイラストから最大23レイヤーに分解

Google Colab **T4 GPU + NF4** で動作するノートブック。
- 初回はモデルのダウンロードに時間がかかります
- **GPU は T4 を選択してください**
- NF4量子化で VRAM ~8GB（T4の16GBに余裕で収まります）

In [ ]:
#@title 1️⃣ セットアップ（GPU確認・リポジトリ・依存関係）
import os, sys, subprocess, shutil
from pathlib import Path
import torch

# GPU確認
if not torch.cuda.is_available():
    raise RuntimeError('⚠️ GPU未検出。ランタイム > ランタイムのタイプを変更 > GPU を選択してください')

gpu_name = torch.cuda.get_device_name(0)
print(f'GPU: {gpu_name}')
print(f'PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}')
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

# リポジトリクローン
REPO_DIR = Path('/content/see-through')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
!git clone --depth 1 https://github.com/shitagaki-lab/see-through.git /content/see-through 2>&1 | tail -3
%cd /content/see-through

# HFキャッシュ設定
os.environ['HF_HOME'] = '/content/.cache/huggingface'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
Path('/content/.cache/huggingface').mkdir(parents=True, exist_ok=True)

# シンボリックリンク
assets_link = REPO_DIR / 'assets'
if assets_link.exists() or assets_link.is_symlink():
    assets_link.unlink()
assets_link.symlink_to(REPO_DIR / 'common' / 'assets', target_is_directory=True)

# 最小限の依存関係インストール
reqs = REPO_DIR / 'requirements-colab.txt'
reqs.write_text('numpy==2.2.6\n'
    'opencv-python==4.13.0.92\n'
    'Pillow==12.1.1\n'
    'pyyaml==6.0.3\n'
    'matplotlib==3.9.4\n'
    'scipy==1.15.3\n'
    'scikit-learn==1.7.2\n'
    'einops==0.8.2\n'
    'transformers==5.0.0\n'
    'diffusers==0.37.0\n'
    'huggingface-hub==1.7.2\n'
    'tokenizers==0.22.2\n'
    'accelerate==1.13.0\n'
    'safetensors==0.7.0\n'
    'bitsandbytes\n'
    'pycocotools==2.0.11\n'
    'psd-tools[composite]==1.14.2\n'
    'tqdm==4.67.3\n', encoding='utf-8')

!pip install -q --upgrade pip setuptools wheel
!pip install -q -r requirements-colab.txt
!pip install -q -e ./common --no-deps

print('\n✅ セットアップ完了！')


In [ ]:
#@title 2️⃣ Headwearパッチ適用（帽子・角の認識修正）
from pathlib import Path

utils_file = Path('/content/see-through/common/utils/inference_utils.py')
lines = utils_file.read_text(encoding='utf-8').split('\n')

patched = False
for i, line in enumerate(lines):
    if 'w // 5' in line and 'px = min' in line:
        start = i
        while start > 0 and '_crop_head' not in lines[start]:
            start -= 1
        end = i + 1
        while end < len(lines) and 'return img[y1' not in lines[end]:
            end += 1
        end += 1
        indent = '        '
        new_func = [
            f'{indent}def _crop_head(img, xywh):',
            f'{indent}    x, y, w, h = xywh',
            f'{indent}    ih, iw = img.shape[:2]',
            f'{indent}    pad_x = int(w * 0.30)',
            f'{indent}    pad_y_up = int(h * 0.60)',
            f'{indent}    pad_y_down = int(h * 0.30)',
            f'{indent}    x1 = max(x - min(pad_x, x), 0)',
            f'{indent}    x2 = min(x + w + min(pad_x, iw - (x + w)), iw)',
            f'{indent}    y1 = max(y - min(pad_y_up, y), 0)',
            f'{indent}    y2 = min(y + h + min(pad_y_down, ih - (y + h)), ih)',
            f'{indent}    return img[y1: y2, x1: x2], (x1, y1, x2, y2)',
        ]
        lines[start:end] = new_func
        patched = True
        break

if patched:
    utils_file.write_text('\n'.join(lines), encoding='utf-8')
    print('✅ headwear crop パッチ適用')
else:
    print('ℹ️ headwear: パッチ済みまたはコード変更あり')


In [ ]:
#@title 3️⃣ 画像アップロード
from google.colab import files
import shutil, os
from pathlib import Path

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('画像がアップロードされませんでした')

input_dir = Path('/content/input')
input_dir.mkdir(parents=True, exist_ok=True)

for name in uploaded:
    dst = input_dir / name
    shutil.move(name, str(dst))
    print(f'✅ {dst}')

INPUT_IMAGE = str(input_dir / next(iter(uploaded.keys())))
print(f'\n▶ 入力画像: {INPUT_IMAGE}')


In [ ]:
#@title 4️⃣ 推論実行（NF4 + T4対策）
import os, time
os.environ['HF_HOME'] = '/content/.cache/huggingface'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

if 'INPUT_IMAGE' not in globals():
    raise RuntimeError('先に画像アップロードセルを実行してください')

# T4対策: monkey-patch付きラッパー作成
# NF4テキストエンコーダをfp16に復元してからcache_tag_embedsを実行
wrapper = r'''
import sys, os
sys.path.insert(0, '/content/see-through')
sys.path.insert(0, '/content/see-through/inference/scripts')
os.chdir('/content/see-through')

import torch
from modules.layerdiffuse.diffusers_kdiffusion_sdxl import KDiffusionStableDiffusionXLPipeline

_orig_cache = KDiffusionStableDiffusionXLPipeline.cache_tag_embeds
def _patched_cache(self):
    for name in ['text_encoder', 'text_encoder_2']:
        enc = getattr(self, name, None)
        if enc is not None and hasattr(enc, 'dequantize'):
            print(f'  [T4 fix] Dequantizing {name} to fp16...')
            setattr(self, name, enc.dequantize().to(device='cuda', dtype=torch.float16))
    result = _orig_cache(self)
    torch.cuda.empty_cache()
    return result
KDiffusionStableDiffusionXLPipeline.cache_tag_embeds = _patched_cache
print('[T4 fix] cache_tag_embeds monkey-patch applied')

exec(open('inference/scripts/inference_psd_quantized.py').read())
'''

with open('/content/run_nf4.py', 'w') as f:
    f.write(wrapper)

t0 = time.time()
!python /content/run_nf4.py \
    --srcp "{INPUT_IMAGE}" \
    --save_dir /content/output \
    --quant_mode nf4 \
    --resolution 1280 \
    --resolution_depth 720 \
    --num_inference_steps 30 \
    --save_to_psd
elapsed = time.time() - t0
print(f'\n⏱️ 処理時間: {elapsed:.1f}秒 ({elapsed/60:.1f}分)')


In [ ]:
#@title 5️⃣ 結果プレビュー
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

img_name = Path(INPUT_IMAGE).stem
out_dir = Path('/content/output') / img_name
if not out_dir.exists():
    raise RuntimeError(f'出力ディレクトリが見つかりません: {out_dir}')

pngs = [p for p in sorted(out_dir.glob('*.png'))
        if '_depth' not in p.name and not p.name.startswith('src_')]

cols = min(6, len(pngs))
rows = (len(pngs) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3), squeeze=False)

for ax, png in zip(axes.flatten(), pngs):
    ax.imshow(Image.open(png))
    ax.set_title(png.stem, fontsize=8)
    ax.axis('off')
for ax in axes.flatten()[len(pngs):]:
    ax.axis('off')

plt.suptitle(img_name, fontsize=14)
plt.tight_layout()
plt.show()

# stats.json表示
stats_path = out_dir / 'stats.json'
if stats_path.exists():
    import json
    print(json.dumps(json.loads(stats_path.read_text()), indent=2))

print('\n▶ 出力ファイル:')
for p in sorted(out_dir.iterdir()):
    print(f'  {p.name} ({p.stat().st_size // 1024}KB)')


In [ ]:
#@title 6️⃣ PSDダウンロード
from pathlib import Path
from google.colab import files

psd_files = sorted(Path('/content/output').glob('**/*.psd'))
if not psd_files:
    raise RuntimeError('PSDが見つかりません。推論セルを先に実行してください。')

for p in psd_files:
    print(f'⬇️ {p.name}')
    files.download(str(p))
